# Contagion engine: stress test demo

1. Set a macro shock in natural units.
2. Read the ranked table: the first row is the company with the largest equity loss.
3. Select a ticker and read its audit trail.

Before you run this notebook, build the data and train the model (see `README.md`).

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
root = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(root); sys.path.insert(0, root)
import pandas as pd
from src.simulator.stress_test import simulate
from src.knowledge.audit import Auditor
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 1. The shock

- `SELIC_delta`: percentage points of the SELIC target. `3.0` is +300 basis points.
- `IPCA_delta`: percentage points of the quarterly IPCA change.
- `PTAX_delta_pct`: percent change of the USD/BRL rate. `15` is +15 %.

The shock starts after the last filed quarter and holds for 4 quarters.

In [ ]:
shocks = {'SELIC_delta': 3.0, 'IPCA_delta': 0.0, 'PTAX_delta_pct': 15.0}
scenario = simulate(shocks, horizon=4)
table = scenario.table()
columns = ['rank', 'ticker', 'sector', 'equity_last', 'equity_base', 'equity_shock', 'equity_change_pct']
table[columns].head(15)

## 2. Checks

Each predicted balance sheet must balance. A zero shock must give a zero change.

In [ ]:
print('largest balance residual / assets:', f"{table['balance_residual'].max():.2e}")
zero = simulate({}, horizon=4).table()
print('largest change with a zero shock (%):', float(zero['equity_change_pct'].abs().max()))

## 3. The audit trail of 1 company

Each line says where its numbers come from: `input`, `filed`, `extracted`, `model` or `estimate`.

In [ ]:
auditor = Auditor(scenario)
ticker = table.iloc[0]['ticker']  # change this to any ticker of the table
print(auditor.render(ticker))

In [ ]:
print(auditor.render('PETR4'))

## Limits

- The shock response is an extrapolation of the model, not a causal estimate.
- The universe holds companies that have all quarters from 2010-Q4 to 2024-Q4.
- An attention weight is a model weight. It is not proof of a causal link.
- The extraction covers only what a company discloses.